# Project Starter: Option 4, Annual-Report Risk Factors (SEC EDGAR 10-K, advanced)

**Data:** the "Risk Factors" section (Item 1A) of U.S. companies' annual reports (Form 10-K), from SEC EDGAR. You download the text yourself. The default route is one year of **EDGAR-CORPUS**, a public copy of 10-K filings already cut into sections. The other route is 10-K documents you save from EDGAR yourself (`.htm` files); Appendix A turns them into a file the notebook can read. The dataset guide (`dataset_option_4_sec_10k_risk_factors.md`) explains both.

This option is **advanced**. A risk-factor section is long (often 5,000 to 15,000 words) and written by lawyers. There are no labels, so most projects make their own: a judged search, a risk category or an extracted field.

| Part | What it does |
|---|---|
| 1 Setup | Packages, folders and your project settings |
| 2 Load and provenance | Read the filings, keep Item 1A, cut it into paragraphs, record where the data came from |
| 3 Audit | Filings, empty sections, paragraph length, repeated boilerplate, personal details |
| 4 Protected split | Split by company, save the paragraph table, record the test IDs, add a guard; samples to label |
| 5 Evaluation plan | Write down the plan before you see any results |
| 6 First baseline | A TF-IDF search you judge yourself, and a keyword rule |
| 7 Save | Save the outputs and the reproducibility record, then commit |
| Appendix A | Only for saved `.htm` files: cut out Item 1A and check the cut |

**How to use it.**

1. Download one year of EDGAR-CORPUS into `project/data/private/edgar/` (the notebook makes this folder in Part 1). For your own `.htm` files, run Part 1 and then Appendix A first.
2. Copy this file into your project folder, `project/notebooks/`, and give it a clear name, such as `01_data_audit_and_split.ipynb`.
3. Fill in the settings in Part 1, then run all cells from the top (**Restart**, then **Run All**). Reading a 200 MB file takes about a minute. Rerunning is always safe: the notebook never overwrites your label or judgment files.
4. Cells marked **Your turn** are for your own work, with a notes cell below each. The notebook runs without them.
5. Commit the executed notebook with MS1. The MS1 brief lists everything else your repository must hold.

This starter has no points. The milestone rubrics in Canvas describe how the project is graded.

## Part 1 Setup

### 1.1 Packages

Select the `ban6403` kernel, then run the cell. The tools for the reproducibility record (hashes and package versions) are loaded at the end, in the last part.

In [ ]:
from pathlib import Path
import json
import re
import textwrap
import unicodedata
import pandas as pd

### 1.2 The folders

Run this notebook from `project/notebooks/`. The repository is then two folders up, `../..`, and every path below starts from there. This notebook uses four places; the cell makes them if they do not exist yet:

- `project/outputs/option4_setup/`: audit tables, the test IDs, the evaluation plan and the manifest. Commit them.
- `project/data/sample/`: the paragraph table with its split (`risk_paragraphs.csv`). Commit it, so the instructor can rerun your work. It must stay under 50 MB; with many filings, keep fewer.
- `project/data/annotations/`: the templates and your label and judgment files. Commit them.
- `project/data/private/edgar/`: the files you downloaded. They are large, so the repository's `.gitignore` keeps `project/data/private/` out of Git.

In [ ]:
REPO = Path("../..")   # this notebook runs from project/notebooks/, two folders below the repository
OUT = REPO / "project/outputs/option4_setup"
SAMPLE_DIR = REPO / "project/data/sample"
ANNOT = REPO / "project/data/annotations"
PRIVATE = REPO / "project/data/private"
# mkdir makes a folder; parents=True also makes missing parent folders; exist_ok=True skips a folder that exists.
for folder in [OUT, SAMPLE_DIR, ANNOT, PRIVATE / "edgar"]:
    folder.mkdir(parents=True, exist_ok=True)

### 1.3 Your project settings

| Setting | What to enter |
|---|---|
| `DECISION_OWNER` | The person or team who will act on your result, such as "Equity research analyst covering retail" |
| `BUSINESS_QUESTION` | The decision in one sentence |
| `TASK` | `"search"` if you will judge search results (Lab 6), or `"label"` if you will label paragraphs yourself (Labs 8 and 9) |
| `CORPUS_FILE` | The EDGAR-CORPUS year file you downloaded, or the file Appendix A made from your `.htm` files |
| `CIKS` | Optional: the companies to keep, by CIK number (EDGAR's company ID), such as `["320193"]`. An empty list `[]` keeps all. |
| `N_FILINGS` | How many filings to keep (drawn with your seed). 150 filings give several thousand paragraphs. |
| `MIN_SECTION_WORDS` | Skip filings whose Item 1A is shorter than this (small companies may leave it out) |
| `MIN_PARAGRAPH_WORDS` | Shorter lines are headings or fragments and are dropped |
| `QUERY` | A first search query for the baseline in Part 6, such as `"cybersecurity breach of customer data"` |

**The data roles follow `TASK`.** For a search project, you tune your queries on `valid` and judge the frozen search on `test` once in MS3. For a label project, you build and tune on a labeled development sample from `train`. A held-out sample from `valid` is then your MS3 evaluation set, and `valid` is not used for tuning.

In [ ]:
DECISION_OWNER = ""
BUSINESS_QUESTION = ""
TASK = "search"           # "search" or "label"
CORPUS_FILE = "project/data/private/edgar/edgar_corpus_2020_test.jsonl"
CIKS = []
N_FILINGS = 150
MIN_SECTION_WORDS = 500
MIN_PARAGRAPH_WORDS = 30
QUERY = "cybersecurity breach of customer data"
SEED = 6403               # fixes every random draw, so each run gives the same samples

# OWN_LABEL is True when you label paragraphs yourself.
OWN_LABEL = TASK == "label"
SETTINGS = {"decision_owner": DECISION_OWNER, "business_question": BUSINESS_QUESTION, "task": TASK,
            "corpus_file": CORPUS_FILE, "ciks": CIKS, "n_filings": N_FILINGS,
            "min_section_words": MIN_SECTION_WORDS, "min_paragraph_words": MIN_PARAGRAPH_WORDS, "query": QUERY}

# Print each setting; {name:>20} right-aligns the name. An empty setting is marked CHECK.
for name, value in SETTINGS.items():
    if value in ("", []):
        value = "CHECK: not filled in yet"
    print(f"{name:>20}: {value}")

## Part 2 Load the data and record its provenance

### 2.1 Read the filings and keep Item 1A

EDGAR-CORPUS files hold one filing per line, with each 10-K item in its own field. The cell reads the file line by line and keeps only the company ID (`cik`), the year, the file name and Item 1A, so memory stays small. Appendix A writes your own `.htm` filings in the same format, so this cell reads them too.

In [ ]:
corpus_path = REPO / CORPUS_FILE   # a full path stays as it is
rows = []
# Read the file one line (one filing) at a time; json.loads turns a line into a dictionary.
with open(corpus_path, encoding="utf-8") as handle:
    for line in handle:
        if line.strip():
            filing = json.loads(line)
            section = filing.get("section_1A")
            if section is None:
                section = ""
            rows.append({"cik": str(filing["cik"]), "year": str(filing["year"]),
                         "filename": filing["filename"], "section": section})
filings_all = pd.DataFrame(rows)
# str.split() cuts each section into words; str.len() counts them.
filings_all["section_words"] = filings_all["section"].str.split().str.len()
print("filings read:", len(filings_all), "| companies:", filings_all["cik"].nunique(),
      "| years:", sorted(filings_all["year"].unique())[:10])

### 2.2 Keep your filings

The cell keeps the companies in `CIKS` (or all), drops filings with a short or missing Item 1A, and draws `N_FILINGS` with your seed. It sorts the filings by file name before the draw, so the same file and settings always give the same filings.

In [ ]:
# Keep the companies in CIKS (written as text), or all companies when CIKS is empty.
if CIKS:
    cik_texts = []
    for cik in CIKS:
        cik_texts.append(str(cik))
    kept = filings_all[filings_all["cik"].isin(cik_texts)]
else:
    kept = filings_all
too_short = int(kept["section_words"].lt(MIN_SECTION_WORDS).sum())
kept = kept[kept["section_words"].ge(MIN_SECTION_WORDS)]
# Sort by file name, then draw N_FILINGS with the seed (min(...) never asks for more than there are).
kept = kept.sort_values("filename")
filings = kept.sample(min(N_FILINGS, len(kept)), random_state=SEED)
filings = filings.sort_values(["cik", "year"]).reset_index(drop=True)
print("filings with Item 1A shorter than", MIN_SECTION_WORDS, "words (dropped):", too_short)
print("filings kept:", len(filings), "| companies:", filings["cik"].nunique())

### 2.3 Cut Item 1A into paragraphs

A whole section is too long to label or to search well. The unit of analysis here is **one paragraph** of the risk-factor section. Lines shorter than `MIN_PARAGRAPH_WORDS` are headings or fragments and are dropped. Each paragraph gets a stable ID: the filing's file name (company and year) and the paragraph's position, such as `320193_2020-p007`. The file name keeps two filings of one company apart.

In [ ]:
# Cut each Item 1A into paragraphs (one per line) and keep those with at least MIN_PARAGRAPH_WORDS words.
rows = []
# itertuples gives one filing at a time; f.section is its Item 1A text.
for f in filings.itertuples(index=False):
    filing_id = Path(f.filename).stem   # the file name without .htm, such as 320193_2020
    position = 0
    for line in f.section.split("\n"):
        # split() and " ".join collapse runs of spaces into one space.
        line = " ".join(line.split())
        if len(line.split()) >= MIN_PARAGRAPH_WORDS:
            position += 1
            # {position:03d} writes 7 as 007.
            rows.append({"record_id": f"{filing_id}-p{position:03d}", "cik": f.cik, "year": f.year,
                         "filename": f.filename, "position": position, "text": line})
data = pd.DataFrame(rows)
data["n_words"] = data["text"].str.split().str.len()
# Every paragraph ID must be unique, because later files join on it.
if data["record_id"].duplicated().any():
    raise ValueError("Two paragraphs share an ID. Check for two copies of the same filing.")
# groupby("filename").size() counts the paragraphs of each filing.
print("paragraphs:", len(data), "| per filing (median):", int(data.groupby("filename").size().median()),
      "| words per paragraph (median):", int(data["n_words"].median()))

### 2.4 Provenance record

Read this record, correct anything that does not fit, and copy it into your data dictionary (`project/docs/data_dictionary.md`). Fill in the date you downloaded the data. If you used Appendix A, change the source to "10-K documents saved from SEC EDGAR (sec.gov)".

In [ ]:
provenance = {
    "source": "EDGAR-CORPUS (Loukas et al., 2021), a copy of SEC EDGAR 10-K filings cut into items",
    "downloaded_on": "",   # the date you downloaded the files
    "terms": "public filings; the SEC allows copying with credit; EDGAR-CORPUS is listed as Apache-2.0",
    "collection": "annual reports that companies must file with the SEC",
    "filings": f"{len(filings)} filings of {filings['cik'].nunique()} companies; years {', '.join(sorted(filings['year'].unique()))}",
    "unit_of_analysis": "one paragraph of Item 1A (Risk Factors) of one 10-K filing",
    "stable_id": "record_id = file name (CIK_YEAR) and paragraph position, such as 320193_2020-p007",
    "personal_details": "company documents; they name executives and directors, who are public figures in this role",
    "may_be_shared": "yes; the downloads are large, so they stay in project/data/private/; the paragraph table is committed",
}

# Print each entry; an empty one is marked CHECK.
for key, value in provenance.items():
    if value in ("", None):
        value = "CHECK: fill in"
    print(f"{key:>18}: {value}")

## Part 3 Audit the corpus

### 3.1 Length, repeats and boilerplate

Many companies use the same standard sentences ("boilerplate"). Repeated paragraphs inflate counts and can sit in two splits at once. The **text key** ignores case, punctuation and extra spaces, so standard paragraphs used by several companies get the same key. The column `unit` says what each row counts, and `of` gives the total of that unit.

In [ ]:
def text_key(text):
    """Take one text; return a key that ignores case, punctuation and extra spaces, in any alphabet."""
    # NFKC writes look-alike characters the same way; casefold() is lower() for every alphabet.
    text = unicodedata.normalize("NFKC", text).casefold()
    # \W+ is a run of characters that are not letters or digits; accented and non-Latin letters count as letters.
    return re.sub(r"\W+", " ", text).strip()

# map applies text_key to every text in the column.
data["text_key"] = data["text"].map(text_key)
# A text of only symbols or emoji has an empty key. Give it its own key, so such texts do not count as copies.
empty_key = data["text_key"].eq("")
data.loc[empty_key, "text_key"] = "empty-" + data.loc[empty_key, "record_id"]

# For each text key: how many different companies use it.
companies_per_key = data.groupby("text_key")["cik"].nunique()
n_paragraphs = len(data)
n_keys = data["text_key"].nunique()
audit = pd.DataFrame([
    {"check": "filings read", "count": len(filings_all), "unit": "filings", "of": len(filings_all)},
    {"check": "filings with Item 1A shorter than the minimum (dropped)", "count": too_short, "unit": "filings", "of": len(filings_all)},
    {"check": "paragraphs", "count": n_paragraphs, "unit": "paragraphs", "of": n_paragraphs},
    {"check": "repeated paragraphs: copies beyond the first", "count": int(data["text_key"].duplicated().sum()), "unit": "paragraphs", "of": n_paragraphs},
    {"check": "paragraph texts used by more than one company", "count": int(companies_per_key.gt(1).sum()), "unit": "distinct texts", "of": n_keys},
])
# describe() gives the count, mean, quartiles (25%, 50% = median, 75%) and more; keep six of them.
length_table = data["n_words"].describe()[["count", "min", "25%", "50%", "75%", "max"]].round(1)
print(audit.to_string(index=False), end="\n\n")
print("words per paragraph:")
print(length_table.to_string())

### 3.2 Personal details

Filings name executives and directors in their public roles. They rarely hold private details, but the scan below checks for emails, phone numbers and long digit strings.

In [ ]:
# Patterns that may be personal details. A match is a reason to read the text, not proof.
# Pattern key: \w a letter or digit, \d a digit, \s a space, \b a word edge, {3} exactly three, + one or more.
PII_PATTERNS = {
    "email address": r"[\w.+-]+@[\w-]+\.[\w.]+",
    "phone number": r"\(?\b\d{3}\)?[\s.-]\d{3}[\s.-]\d{4}\b",
    "web link": r"https?://\S+|www\.\S+",
    "long digit string (card or account?)": r"\b\d{9,}\b",
    "@handle": r"(?<![\w.])@\w{3,}",
}

def pii_scan(texts):
    """Take a column of texts; return how many texts match each pattern above."""
    rows = []
    for name, pattern in PII_PATTERNS.items():
        # True for each text with at least one match.
        hits = texts.str.contains(pattern, regex=True)
        rows.append({"pattern": name, "texts with a match": int(hits.sum()), "of": len(texts)})
    return pd.DataFrame(rows)

pii_table = pii_scan(data["text"])
pii_table

**Your turn A.** Read the risk-factor paragraphs of two filings from start to end. Note how one risk is introduced, which paragraphs are boilerplate, and two paragraphs that would be hard to label or judge. The kept paragraphs leave out headings and short lines. So open one of the filings in full as well (the EDGAR page, or the `section` column of `filings`), and note what the paragraph cut leaves out.

In [ ]:
# Your turn A
# TODO 1: pick a filing and print its paragraphs in full, in order:
#         one_filing = data[data["filename"].eq(filings["filename"].iloc[0])]
#         for paragraph in one_filing["text"]:
#             print(textwrap.fill(paragraph, 100)); print()
# TODO 2: do the same for a second filing (iloc[1]), and read one whole section: print(filings["section"].iloc[0])
# TODO 3: write your notes in the cell below.

**Your notes (Your turn A)**

- Filings read (file names):
- How a risk is introduced:
- Boilerplate you noticed (paragraph IDs):
- Hard paragraph 1 (ID, why it is hard):
- Hard paragraph 2 (ID, why it is hard):
- What the paragraph cut leaves out:

## Part 4 Make and protect the split

You make the split **once**, save it, and never redraw it. The split is **by company**: all paragraphs of one company, in every year you keep, go to the same split. About 70% of companies go to `train`, 15% to `valid` and 15% to `test`. A method is then never tested on a company it saw in training.

Boilerplate can still repeat across companies. A repeated text stays only in its earliest split, in the order `train`, `valid`, `test`. Later copies are dropped, and their IDs go to `project/outputs/option4_setup/dropped_cross_split_copies.csv`.

The cell then saves the final paragraph table, with its split, to `project/data/sample/risk_paragraphs.csv`. On a later run it compares the result with the saved table and stops if the data or settings changed.

### 4.1 Make the split

In [ ]:
# Shuffle the companies with the seed: sorted() first, so the order does not depend on the input file.
# sample(frac=1) draws all rows in a random order; tolist() makes a plain list.
companies = pd.Series(sorted(data["cik"].unique())).sample(frac=1, random_state=SEED).tolist()
n_companies = len(companies)
# Whole companies go to one split: the first 70% train, the next 15% valid, the rest test.
company_split = {}
for i, cik in enumerate(companies):
    if i < round(0.7 * n_companies):
        company_split[cik] = "train"
    elif i < round(0.85 * n_companies):
        company_split[cik] = "valid"
    else:
        company_split[cik] = "test"
# map looks up each paragraph's company in the dictionary.
data["split"] = data["cik"].map(company_split)

# The text keys found in train, and those found in valid.
train_keys = set(data.loc[data["split"].eq("train"), "text_key"])
valid_keys = set(data.loc[data["split"].eq("valid"), "text_key"])
# isin is True where a key is in the set; & means "and", | means "or".
valid_copy = data["split"].eq("valid") & data["text_key"].isin(train_keys)
test_copy = data["split"].eq("test") & (data["text_key"].isin(train_keys) | data["text_key"].isin(valid_keys))
is_copy = valid_copy | test_copy
dropped_copies = data.loc[is_copy, ["record_id", "split"]]
dropped_copies.to_csv(OUT / "dropped_cross_split_copies.csv", index=False)
# ~ means "not": keep the records that are not dropped copies.
data = data[~is_copy].copy()
print("copies dropped from valid:", int(valid_copy.sum()), "| from test:", int(test_copy.sum()),
      "| paragraphs left:", len(data))

PARAGRAPHS = SAMPLE_DIR / "risk_paragraphs.csv"
COLUMNS = ["record_id", "cik", "year", "filename", "position", "split", "n_words", "text"]
if PARAGRAPHS.exists():
    saved_table = pd.read_csv(PARAGRAPHS, dtype={"record_id": str, "cik": str, "year": str})
    # The same paragraphs in the same order, with the same splits?
    same_ids = saved_table["record_id"].tolist() == data["record_id"].tolist()
    same_splits = saved_table["split"].tolist() == data["split"].tolist()
    if not (same_ids and same_splits):
        raise ValueError("The data or settings changed since the paragraph table was saved. Restore them, "
                         "or ask the instructor before you make a new split.")
    print("The paragraphs and the split match the saved table:", PARAGRAPHS.name)
else:
    data[COLUMNS].to_csv(PARAGRAPHS, index=False)
    print("Saved the paragraph table with its split:", PARAGRAPHS.name)
# The file size in MB (stat().st_size is in bytes). GitHub warns above 50 MB.
size_mb = PARAGRAPHS.stat().st_size / 1e6
print(f"{PARAGRAPHS.name}: {size_mb:.1f} MB")
if size_mb >= 50:
    print("CHECK: above 50 MB. Keep fewer filings before you commit it.")
# Paragraphs and companies per split; reindex keeps all three splits in order.
per_split = data.groupby("split").agg(paragraphs=("record_id", "size"), companies=("cik", "nunique"))
per_split.reindex(["train", "valid", "test"], fill_value=0)

### 4.2 Record the test IDs and a guard

The test IDs go to `project/outputs/option4_setup/protected_test_ids.csv`; their fingerprint goes into the manifest at the end. The guard `check_protected` stops a cell if a table you are about to fit or tune on holds a protected record. Protected means a test record, and, if you make your own label, a record of your held-out sample. Call the guard before every fit until MS3.

In [ ]:
test_ids = data.loc[data["split"].eq("test"), "record_id"].tolist()
# Save the test IDs, so you can show later that no method used them before MS3.
pd.DataFrame({"record_id": sorted(test_ids)}).to_csv(OUT / "protected_test_ids.csv", index=False)
# A set of protected IDs; the held-out sample is added to it if you make your own label.
protected_ids = set(test_ids)

def check_protected(frame, id_col="record_id"):
    """Take a table; stop with an error if any of its IDs is protected (test, or your held-out sample)."""
    # & keeps the IDs that are in both sets.
    overlap = set(frame[id_col]) & protected_ids
    if overlap:
        raise ValueError(f"{len(overlap)} protected records found. Do not fit or tune on them before MS3.")
    print(f"ok: no protected records among {len(frame):,} rows")

print("protected test paragraphs:", len(test_ids))
check_protected(data[data["split"].ne("test")])

### 4.3 Samples to label (only for your own label)

Many projects need a label that the data does not have, such as "describes a cybersecurity risk". Then set `TASK = "label"` in Part 1.3, and read and label records yourself with your annotation guide. Every own-label project follows the same route:

- The **development sample** comes from `train`. Use it to write and test your guide, and to build and tune your methods.
- The **held-out sample** comes from `valid`. It is your evaluation set for MS3. Label it with the finished guide by the MS2 freeze, before any method sees it. With your own label, `valid` is not used for tuning, and the guard blocks the held-out records.
- Neither sample comes from `test`, so you never read a test record before MS3.

**How the files work.** The cell writes two blank templates to `project/data/annotations/`: `dev_labels_template.csv` and `heldout_labels_template.csv`. Copy each one to `dev_labels.csv` and `heldout_labels.csv`, and fill the `own_label` column in the copies. The notebook only reads these two files, so a rerun never erases your labels.

A template is written only when neither it nor your copy exists. To get a fresh template, for example after you change a sample size, delete the old template first. Do not use `NA` or `None` as a label value: pandas reads them as empty. Risk paragraphs are dense: plan two to three hours for 100.

In [ ]:
N_DEV_LABEL = 100
N_HELDOUT_LABEL = 100
DEV_TEMPLATE = ANNOT / "dev_labels_template.csv"
HELDOUT_TEMPLATE = ANNOT / "heldout_labels_template.csv"
DEV_LABELS = ANNOT / "dev_labels.csv"           # your labeled copy: the notebook only reads it
HELDOUT_LABELS = ANNOT / "heldout_labels.csv"   # your labeled copy: the notebook only reads it

def label_sample(split, n):
    """Take a split name and a size; return a seeded sample of that split with empty own_label and note columns."""
    pool = data[data["split"].eq(split)]
    # min(n, len(pool)) never asks for more rows than the split has; random_state=SEED fixes the draw.
    sample = pool.sample(min(n, len(pool)), random_state=SEED)
    sample = sample[["record_id", "cik", "year", "text"]].sort_values("record_id")
    sample["own_label"] = ""
    sample["note"] = ""
    return sample

def write_template(sample, template, completed):
    """Save a blank template, but only if neither the template nor your completed file exists yet."""
    if template.exists() or completed.exists():
        print("kept as it is:", template.name, "and", completed.name)
    else:
        sample.to_csv(template, index=False)
        print("wrote", template.name)

heldout_ids = []
if OWN_LABEL:
    dev_sample = label_sample("train", N_DEV_LABEL)
    heldout_sample = label_sample("valid", N_HELDOUT_LABEL)
    write_template(dev_sample, DEV_TEMPLATE, DEV_LABELS)
    write_template(heldout_sample, HELDOUT_TEMPLATE, HELDOUT_LABELS)
    heldout_ids = heldout_sample["record_id"].tolist()
    # From now on the guard also blocks the held-out records.
    protected_ids.update(heldout_ids)
    print("development sample (train):", len(dev_sample), "| held-out sample (valid):", len(heldout_ids))
else:
    print(f"TASK is '{TASK}', so you do not need samples to label.")

### 4.4 Load your labels

Once you have labeled records in `dev_labels.csv`, this cell reads them and joins them to the data by `record_id`. A **join** (pandas `merge`) adds the columns of one table to the rows of another table that have the same ID. After the join, the column `own_label` holds your label for the records you labeled; it is empty for all others. The cell only counts the held-out labels. They stay unused until MS3.

In [ ]:
if OWN_LABEL and DEV_LABELS.exists():
    # Read the IDs as text, so the join matches them exactly (an ID such as 00123 keeps its zeros).
    dev_labels = pd.read_csv(DEV_LABELS, dtype={"record_id": str})
    # Keep the ID and the label, and drop the rows you have not labeled yet.
    dev_labels = dev_labels[["record_id", "own_label"]].dropna(subset=["own_label"])
    # merge joins the labels to data by ID; how="left" keeps every record of data, labeled or not.
    # drop(...) first removes an own_label column from an earlier run of this cell.
    data = data.drop(columns="own_label", errors="ignore").merge(dev_labels, on="record_id", how="left")
    labeled = data[data["own_label"].notna()]
    print("labeled development records:", len(labeled), "| outside train (should be 0):", int(labeled["split"].ne("train").sum()))
    print(labeled["own_label"].value_counts().to_string())
elif OWN_LABEL:
    print("No dev_labels.csv yet. Copy the template, label some records, then run the notebook again.")
if OWN_LABEL and HELDOUT_LABELS.exists():
    heldout_labels = pd.read_csv(HELDOUT_LABELS, dtype={"record_id": str})
    # The IDs in your held-out file are protected too, even if you later change the sample size.
    protected_ids.update(heldout_labels["record_id"])
    n_done = int(heldout_labels["own_label"].notna().sum())
    print(f"held-out labels filled in: {n_done} of {len(heldout_labels)} (finish them by the MS2 freeze)")

## Part 5 Write the evaluation plan now

Decide how you will judge success before you see any result. Then nobody, including you, can move the target after the fact. Fill in the plan. The cell flags any field that is still empty. The MS1 brief has a metric menu: a main metric and a baseline for each type of task, and the module that teaches it. The main metric and the success threshold are **provisional until MS2**: you may change them once, with a reason, when you freeze the final test plan.

The two data fields are filled in from `TASK`, so the saved plan always matches what the notebook does. Do not type over them.

In [ ]:
# The data roles follow TASK.
if OWN_LABEL:
    data_for_tuning = "own label: the labeled development sample from train only; valid is not used for tuning"
    data_for_final_check = (f"own label: the held-out sample from valid ({len(heldout_ids)} paragraphs), "
                            "labeled before the MS2 freeze and scored once in MS3")
else:
    data_for_tuning = "train to fit TF-IDF; valid to judge the top results and tune the queries"
    data_for_final_check = "test: the frozen queries are run once in MS3, and the top results judged"

EVALUATION_PLAN = {
    "task": "",               # for example: find the paragraphs that describe a cybersecurity risk
    "output": "",             # what the decision owner receives
    "target": TASK,
    "baseline": "",           # the simple method you must beat, for example: a keyword rule, then TF-IDF search
    "candidate_methods": "",  # one or two methods you may compare in MS3
    "main_metric": "",        # see the metric menu in the MS1 brief; for example: precision at 10
    "success_criterion": "",  # the business threshold (provisional until MS2)
    "data_for_tuning": data_for_tuning,
    "data_for_final_check": data_for_final_check,
    "uncertainty": "",        # for example: k of n now; a paired comparison with its bootstrap interval in MS3 (Lab 13)
}
# One row per field of the plan: ok, or CHECK when it is still empty.
check_rows = []
for field, value in EVALUATION_PLAN.items():
    if value.strip():
        status = "ok"
    else:
        status = "CHECK: empty"
    check_rows.append({"field": field, "status": status})
plan_check = pd.DataFrame(check_rows)
(OUT / "evaluation_plan.json").write_text(json.dumps(EVALUATION_PLAN, indent=2))
plan_check

## Part 6 A first baseline

Risk paragraphs have no labels, so the first baseline is a **search**, as in Lab 6. TF-IDF is fitted on `train` and ranks paragraphs for your `QUERY`. For a search project it ranks the `valid` paragraphs. For a label project it ranks `train` paragraphs instead, because `valid` holds your held-out sample; use the results to find examples for your guide. The cell prints the top 10 in full and saves them as a blank judgment template, `search_judgments_template.csv`, in `project/data/annotations/`.

A **keyword rule** gives a second, simpler baseline: the share of companies whose paragraphs mention your topic.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

# A search project ranks valid; a label project ranks train, so the held-out sample stays unseen.
if OWN_LABEL:
    search_split = "train"
else:
    search_split = "valid"
train = data[data["split"].eq("train")]
pool = data[data["split"].eq(search_split)]
check_protected(train)
check_protected(pool)
# TF-IDF fitted on train only; sublinear_tf=True uses 1 + log(count), so a word repeated many times counts less.
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2, sublinear_tf=True)
vectorizer.fit(train["text"])
# linear_kernel gives the cosine between the query and each paragraph (TF-IDF rows have length 1).
# ravel() turns the one-row result into a plain list of scores.
scores = linear_kernel(vectorizer.transform([QUERY]), vectorizer.transform(pool["text"])).ravel()
found = pool.copy()
found["score"] = scores   # full precision, so ties are real ties
# Keep paragraphs that share at least one term with the query, best first; at most 10.
found = found[found["score"].gt(0)]
top = found.sort_values("score", ascending=False).head(10)
print(f"top {len(top)} {search_split} paragraphs for: {QUERY}\n")
for row in top.itertuples(index=False):
    # :.3f rounds the score to three decimals for reading only.
    print(f"{row.record_id} | company {row.cik} | score {row.score:.3f}")
    print(textwrap.fill(row.text, 100))
    print()
# A blank judgment template, written only if neither it nor your judged copy exists.
JUDGE_TEMPLATE = ANNOT / "search_judgments_template.csv"
JUDGMENTS = ANNOT / "search_judgments.csv"   # your judged copy: the notebook only reads it
to_judge = top[["record_id", "cik", "year", "score", "text"]].copy()
to_judge["relevant"] = ""
write_template(to_judge, JUDGE_TEMPLATE, JUDGMENTS)

In [ ]:
KEYWORDS = r"cyber|data breach|hack|ransomware"
# case=False ignores upper and lower case; True for each paragraph with a match.
data["keyword_hit"] = data["text"].str.contains(KEYWORDS, case=False, regex=True)
# Development splits only: a label project keeps valid unseen.
if OWN_LABEL:
    keyword_splits = ["train"]
else:
    keyword_splits = ["train", "valid"]
rows = []
for split_name in keyword_splits:
    in_split = data[data["split"].eq(split_name)]
    with_hit = in_split[in_split["keyword_hit"]]
    rows.append({"split": split_name, "companies": in_split["cik"].nunique(),
                 "companies with a hit": with_hit["cik"].nunique(),
                 "paragraphs": len(in_split), "paragraphs with a hit": len(with_hit)})
keyword_table = pd.DataFrame(rows)
keyword_table.to_csv(OUT / "first_keyword_rule.csv", index=False)
keyword_table

**Your turn B.** Judge the search results with a written relevance rule for your topic.

1. Copy `search_judgments_template.csv` to `search_judgments.csv` in `project/data/annotations/`.
2. Read each paragraph in full, and write `yes` or `no` in the column `relevant`. Use only these two values.
3. Run the notebook again. The next cell reports precision at 10 as k of n: k relevant among the n results you judged. If the search returns fewer than 10 paragraphs, report k of that smaller number and say so.
4. Then change `QUERY` or `KEYWORDS` to your own topic. A new query needs new judgments: delete the old template, and rename your old judged file so you keep it.

In [ ]:
# Your turn B
# TODO 1: copy search_judgments_template.csv to search_judgments.csv and fill the relevant column (yes / no).
# TODO 2: write your relevance rule in the cell below, with one example of each value.
# TODO 3: after you change QUERY, keep your old judgments under a new name, such as search_judgments_query1.csv.

In [ ]:
if JUDGMENTS.exists():
    judged = pd.read_csv(JUDGMENTS, dtype={"record_id": str})
    # Only judgments of the current top results count; a judgment of an old query is ignored.
    judged = judged[judged["record_id"].isin(top["record_id"])]
    # astype(str) turns empty cells into the text "nan"; strip() and lower() accept " Yes" as "yes".
    answers = judged["relevant"].astype(str).str.strip().str.lower()
    n_judged = int(answers.isin(["yes", "no"]).sum())
    k_relevant = int(answers.eq("yes").sum())
    print(f"precision at 10: {k_relevant} of {n_judged} judged results are relevant "
          f"({len(top)} results returned)")
else:
    print("No search_judgments.csv yet: see Your turn B.")

**Your notes (Your turn B)**

- Relevance rule (one sentence):
- Example of a relevant paragraph (ID):
- Example of a paragraph that is not relevant (ID):
- Precision at 10 (k of n), and what it means for your question:

## Part 7 Save the outputs

### 7.1 Save the tables

The paragraph table is already in `project/data/sample/`. The outputs folder gets the audit, the ID lists, the plan and the manifest. The cell lists every file this run wrote.

In [ ]:
# The provenance as a two-column table (.T turns the one row into a column), then the audit tables.
pd.DataFrame([provenance]).T.rename(columns={0: "value"}).to_csv(OUT / "provenance.csv")
audit.to_csv(OUT / "audit_text_checks.csv", index=False)
length_table.to_csv(OUT / "audit_paragraph_length.csv")
pii_table.to_csv(OUT / "audit_personal_details_scan.csv", index=False)
filings[["cik", "year", "filename", "section_words"]].to_csv(OUT / "filings_kept.csv", index=False)
# The files written in this run, for the manifest.
saved = [OUT / "provenance.csv", OUT / "audit_text_checks.csv", OUT / "audit_paragraph_length.csv",
         OUT / "audit_personal_details_scan.csv", OUT / "filings_kept.csv", OUT / "dropped_cross_split_copies.csv",
         OUT / "protected_test_ids.csv", OUT / "evaluation_plan.json", OUT / "first_keyword_rule.csv",
         PARAGRAPHS]

### 7.2 Reproducibility record

The manifest is the record a reader uses to check a rerun. It holds your settings, the seed and the package versions. It also holds a SHA-256 hash of each input file, of your label files and of every output of this run. A **SHA-256 hash** is a 64-character code computed from a file's bytes; if one character changes, the hash changes. The **fingerprint** of an ID list is the hash of the sorted IDs; it shows that the protected test set never changed. Module 14 explains these records. You only need to run the two cells.

In [ ]:
from datetime import datetime, timezone
import hashlib
import platform
import matplotlib
import sklearn

def text_sha256(path):
    """Take a file path; return the SHA-256 hash of the file's bytes (Windows line endings read as \n)."""
    file_bytes = Path(path).read_bytes().replace(b"\r\n", b"\n")
    return hashlib.sha256(file_bytes).hexdigest()

def ids_sha256(ids):
    """Take a list of IDs; return one fingerprint of the list (the order of the IDs does not matter)."""
    id_texts = []
    for record_id in ids:
        id_texts.append(str(record_id))
    joined = "\n".join(sorted(id_texts))
    return hashlib.sha256(joined.encode()).hexdigest()

def file_record(path):
    """Take a file path; return its path inside the repository and the first 12 characters of its hash."""
    path = Path(path)
    # relative_to(REPO) drops the "../.." part; as_posix() writes the path with / on every computer.
    return {"file": path.relative_to(REPO).as_posix(), "sha256": text_sha256(path)[:12]}

versions = {"python": platform.python_version(), "pandas": pd.__version__,
            "scikit-learn": sklearn.__version__, "matplotlib": matplotlib.__version__}

# The downloaded corpus file (not committed) and the paragraph table (committed).
input_files = [{"file": corpus_path.name, "sha256": text_sha256(corpus_path)[:12], "committed": False},
               {"file": "project/data/sample/risk_paragraphs.csv", "sha256": text_sha256(PARAGRAPHS)[:12], "committed": True}]

In [ ]:
# The label files, if you have them: their hashes show later that labels did not change after you saw results.
label_files = []
for path in [DEV_LABELS, HELDOUT_LABELS, JUDGMENTS]:
    if path.exists():
        label_files.append(file_record(path))
output_records = []
for path in saved:
    output_records.append(file_record(path))
manifest = {
    "created_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M"),
    "notebook": "sec_10k_project_starter",
    "settings": SETTINGS,
    "seed": SEED,
    "versions": versions,
    "input_files": input_files,
    "protected_test": {"n_ids": len(test_ids), "ids_sha256": ids_sha256(test_ids)},
    "heldout_sample": {"split": "valid", "n_ids": len(heldout_ids), "ids_sha256": ids_sha256(heldout_ids)},
    "label_files": label_files,
    "outputs": output_records,
}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2))
print("protected test fingerprint:", manifest["protected_test"]["ids_sha256"][:12])
print("files saved in this run:")
for record in output_records + [file_record(OUT / "manifest.json")]:
    print(" ", record["file"])

## Commit your work

1. Save this notebook (keep the outputs).
2. In GitHub Desktop, look at the changed files. You should see this notebook and the files in `project/outputs/option4_setup/`, the paragraph table in `project/data/sample/` and your files in `project/data/annotations/`. You should **not** see anything from `project/data/private/`.
3. Write a short commit message, such as "MS1 setup: audit and protected split", then commit and push.
4. Push before the deadline. The instructor grades what your repository holds at the deadline, so a commit that is not pushed does not count.

## Next steps

- **MS1 (Week 6):** use the outputs above in your plan: the provenance record, the audit, the split by company and the evaluation plan. Write your own relevance or label guide for one or two risk topics.
- **MS2 (Week 9):** search projects judge the results on `valid`, as in Lab 6 (relevance judgments, precision at k, reciprocal rank). Label projects train a classifier on the labeled development sample, as in Labs 8 and 9, and check it on part of that sample.
- **MS3 (Week 13):** compare your selected method with the baseline once: frozen queries on `test`, or the held-out sample for a label project. Report k of n and the paired comparison with its bootstrap interval from Lab 13.
- **Final project (Week 15):** Lab 14 adds an interval for each key number and the exhibit standard.
- **Use TF-IDF as your text representation.** The course environment has no embedding model for new texts. Lab 10 themes (NMF) and Lab 11 extraction rules also work on these paragraphs.
- **An LLM is optional.** Filings are public, so a small sample may go to UTSA Copilot Chat or a free API tier. Save every output with the model, version, date and prompt.

## Appendix A Your own 10-K pages (Route B only)

Use this appendix only if you saved 10-K pages from EDGAR as `.htm` files. It turns each page into plain text, cuts out Item 1A, and writes all filings to one file in the EDGAR-CORPUS format. The main notebook then reads that file in Part 2.

**How to run it.** Save the pages in `project/data/private/edgar/html/` as `CIK_YEAR.htm`, such as `320193_2024.htm`. Set `RUN_APPENDIX_A = True` below. Run Part 1, then the two appendix cells (not Run All). Check the cuts, as described below. Then set `CORPUS_FILE = "project/data/private/edgar/my_10k_filings.jsonl"` in Part 1.3 and run the whole notebook.

**How the cut works.** Item 1A runs from the heading "Item 1A. Risk Factors" to the next item ("Item 1B" or "Item 2"). The table of contents names Item 1A too, so the code keeps the longest such stretch. It can still fail on unusual layouts.

**Check the cuts before you use them.** The table shows, for each file, the word count and the first and last words of the cut. Open two or three original pages in a browser. Does each cut start at the Item 1A heading in the body, not in the table of contents? Does it end just before Item 1B or Item 2? Look closely at very short cuts and at cuts marked "no end heading found", which run to the end of the document. Fix or drop those files before you go on.

In [ ]:
from html.parser import HTMLParser

RUN_APPENDIX_A = False   # set to True to turn your saved .htm files into one corpus file
HTML_DIR = PRIVATE / "edgar/html"
HTML_CORPUS = PRIVATE / "edgar/my_10k_filings.jsonl"

class TextOnly(HTMLParser):
    """An HTML reader that keeps the visible text and starts a new line at each block tag."""
    BLOCKS = {"p", "div", "br", "tr", "li", "h1", "h2", "h3", "h4", "h5", "h6", "table"}

    def __init__(self):
        # super().__init__() first sets up the standard HTMLParser; then we add our own two attributes.
        super().__init__()
        self.parts = []  # the pieces of text found so far
        self.skip = 0    # above 0 while inside <script> or <style>, whose content is not visible text

    # HTMLParser calls these three methods for each start tag, end tag and piece of text it reads.
    def handle_starttag(self, tag, attrs):
        if tag in ("script", "style"):
            self.skip += 1
        if tag in self.BLOCKS:
            self.parts.append("\n")

    def handle_endtag(self, tag):
        if tag in ("script", "style"):
            self.skip -= 1
        if tag in self.BLOCKS:
            self.parts.append("\n")

    def handle_data(self, data):
        if not self.skip:
            self.parts.append(data)

def html_to_text(html):
    """Take the HTML of a filing; return its text, one block per line, without empty lines."""
    parser = TextOnly()
    parser.feed(html)
    # \xa0 is a non-breaking space; make it a normal space.
    text = "".join(parser.parts).replace("\xa0", " ")
    lines = []
    for line in text.split("\n"):
        line = " ".join(line.split())
        if line:
            lines.append(line)
    return "\n".join(lines)

# Headings that start and end Item 1A; re.I ignores case, and [.:\-–—]? allows a dot, colon or dash.
START = re.compile(r"item\s*1a\s*[.:\-–—]?\s*risk\s+factors", re.I)
END = re.compile(r"item\s*1b\s*[.:\-–—]?\s*unresolved|item\s*2\s*[.:\-–—]?\s*properties", re.I)

def item_1a(text):
    """Take the text of a filing; return the longest Item 1A stretch and whether an end heading was found."""
    best = ""
    best_has_end = False
    # A table of contents also names Item 1A, so try every start heading and keep the longest piece.
    for start in START.finditer(text):
        end = END.search(text, start.end())
        if end:
            piece = text[start.start():end.start()]
        else:
            piece = text[start.start():]
        if len(piece) > len(best):
            best = piece
            best_has_end = end is not None
    return best, best_has_end

In [ ]:
if RUN_APPENDIX_A:
    files = sorted(HTML_DIR.glob("*.htm*"))
    print("pages found:", len(files))
    cut_rows = []
    with open(HTML_CORPUS, "w", encoding="utf-8") as corpus_out:
        for path in files:
            # partition("_") splits "320193_2024" into "320193", "_", "2024".
            cik, _, year = path.stem.partition("_")
            text = html_to_text(path.read_text(encoding="utf-8", errors="replace"))
            section, has_end = item_1a(text)
            corpus_out.write(json.dumps({"filename": path.name, "cik": cik, "year": year, "section_1A": section}) + "\n")
            words = section.split()
            note = ""
            if not has_end:
                note = "no end heading found"
            cut_rows.append({"file": path.name, "words in cut": len(words), "starts with": " ".join(words[:12]),
                             "ends with": " ".join(words[-12:]), "note": note})
    boundary_check = pd.DataFrame(cut_rows)
    print("wrote", HTML_CORPUS.name)
    print(boundary_check.to_string(index=False))
else:
    print("Appendix A is off (RUN_APPENDIX_A = False).")